# 03. 라벨링 대상 샘플링 & 데이터 분할

정제 데이터(11,177건)에서 **2,000건**을 골라 속성별 감성 라벨을 붙이고, Train / Validation / Test로 나눈다.

**샘플링 전략**
- **무작위 풀 1,000건:** 실제 리뷰 분포를 그대로 반영 → Test 300 / Val 200 / Train 500
- **보강 풀 1,000건:** 무작위로는 거의 안 나오는 속성(사운드·조작감·그래픽 등)을 키워드로 찾아 추가 → **Train에만** 사용
- 보강 샘플을 Test에 넣지 않는 이유: 평가셋이 실제 분포와 달라지면 성능이 과대·과소 평가되기 때문

**Seed = 42** (샘플링·분할 모두)

라벨 기준은 `docs/labeling_guide.md` 참고.

**산출물:** `data/labeling/to_label.csv` (recommendationid, pool, split, text …)

In [1]:
# ── 환경 설정 ────────────────────────────────────────────
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")       # Drive 연결 위치(수정 X)

# ▶ 프로젝트 폴더 경로 (01 노트북과 동일하게)
def find_root():
    """프로젝트 폴더 자동 탐색. Drive의 한글 폴더명이 NFD(자모 분리)로 저장되면
    "ML 기말" 같은 문자열 경로가 일치하지 않으므로, MyDrive를 직접 훑어
    data/processed/reviews_clean.csv가 있는 폴더를 찾는다(검수 파일이 있는 쪽 우선)."""
    if os.environ.get("ML_ROOT"):
        return Path(os.environ["ML_ROOT"])
    base = Path("/content/drive/MyDrive")
    cands = [p for p in [base, *base.glob("*"), *base.glob("*/*")]
             if p.is_dir() and (p / "data" / "processed" / "reviews_clean.csv").exists()]
    if not cands:
        print("⚠ 프로젝트 폴더를 찾지 못해 기본 경로를 사용합니다.")
        return base / "ML 기말"
    cands.sort(key=lambda p: not (p / "data" / "labeling" / "test_verify.csv").exists())
    if len(cands) > 1:
        print("⚠ 프로젝트 폴더 후보가 여러 개입니다 → 첫 번째 사용:", *cands, sep="\n   ")
    return cands[0]

ROOT = find_root()
sys.path.append(str(ROOT / "src"))      # src/preprocess.py 등 공용 모듈
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "| exists:", ROOT.exists())

Mounted at /content/drive
ROOT: /content/drive/MyDrive/머신러닝 포트폴리오 | exists: True


In [2]:
import re
import numpy as np
import pandas as pd

SEED = 42
N_RANDOM, N_BOOST = 1000, 1000
N_TEST, N_VAL = 300, 200

df = pd.read_csv(ROOT / "data" / "processed" / "reviews_clean.csv", dtype={"recommendationid": str})
print(f"정제 데이터: {len(df):,}건")

정제 데이터: 11,177건


## 1. 무작위 풀 (1,000건) → Test / Val / Train 분할

In [3]:
random_pool = df.sample(n=N_RANDOM, random_state=SEED).copy()
random_pool["pool"] = "random"

# 무작위 풀 안에서 다시 섞어 Test 300 / Val 200 / Train 500
order = np.random.RandomState(SEED).permutation(len(random_pool))
split = np.array(["train"] * len(random_pool), dtype=object)
split[order[:N_TEST]] = "test"
split[order[N_TEST:N_TEST + N_VAL]] = "val"
random_pool["split"] = split
random_pool["split"].value_counts()

,count
split,
train,500
test,300
val,200


## 2. 보강 풀 (1,000건) — 희소 속성 키워드 샘플링

무작위 풀을 제외한 나머지에서, 속성별 키워드를 포함한 리뷰를 할당량만큼 뽑는다.

In [4]:
KEYWORDS = {
    "graphics":    r"그래픽|화질|비주얼|풍경|텍스처|아트|광원|모델링|graphic",
    "performance": r"최적화|프레임|fps|FPS|렉|랙|끊김|버벅|사양|로딩|발적화|그래픽카드",
    "story":       r"스토리|서사|엔딩|결말|등장인물|세계관|주인공|퀘스트",
    "controls":    r"조작|키보드|패드|컨트롤|타격감|UI|인터페이스|카메라|키 ?배치|마우스",
    "sound":       r"사운드|음악|bgm|BGM|OST|ost|브금|효과음|성우|더빙|음향|노래",
    "price":       r"가격|가성비|할인|세일|비싸|싸게|과금|현질|DLC|dlc|유료|돈값",
    "operation":   r"버그|튕|크래시|오류|핵쟁이|핵|서버|운영|패치|업데이트|개발사|한글화|번역",
}
QUOTA = {"sound": 180, "controls": 180, "graphics": 160, "performance": 140,
         "story": 120, "price": 120, "operation": 100}
assert sum(QUOTA.values()) == N_BOOST

rest = df[~df["recommendationid"].isin(random_pool["recommendationid"])]
picked, used = [], set()
for i, (aspect, quota) in enumerate(QUOTA.items()):
    cand = rest[rest["text"].str.contains(KEYWORDS[aspect], regex=True)
                & ~rest["recommendationid"].isin(used)]
    take = cand.sample(n=min(quota, len(cand)), random_state=SEED + i)
    used.update(take["recommendationid"])
    picked.append(take.assign(pool="boost", split="train", boost_aspect=aspect))
    print(f"{aspect:<12} 후보 {len(cand):>5}건 → {len(take)}건")

boost_pool = pd.concat(picked)
print("보강 풀:", len(boost_pool))

sound        후보   219건 → 180건
controls     후보   422건 → 180건
graphics     후보   450건 → 160건
performance  후보   705건 → 140건
story        후보  1414건 → 120건
price        후보   699건 → 120건
operation    후보  1132건 → 100건
보강 풀: 1000


## 3. 합치기 및 저장

In [5]:
to_label = pd.concat([random_pool.assign(boost_aspect=""), boost_pool], ignore_index=True)
assert to_label["recommendationid"].is_unique

cols = ["recommendationid", "pool", "split", "boost_aspect", "game", "genre",
        "voted_up", "length", "text"]
to_label = to_label[cols]

OUT = ROOT / "data" / "labeling" / "to_label.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)
if OUT.exists():
    # 이미 라벨이 붙은 파일 → 덮어쓰지 않고 재현성만 확인 (행 순서가 바뀌면 라벨이 어긋나므로)
    prev = pd.read_csv(OUT, dtype={"recommendationid": str})
    same = prev["recommendationid"].tolist() == to_label["recommendationid"].tolist()
    print(f"기존 {OUT.name} 유지 ({len(prev)}건) | 같은 seed로 재샘플링한 결과와 동일: {same}")
    assert same, "샘플링 결과가 기존 파일과 다릅니다. 라벨과 어긋나므로 기존 파일을 사용하세요."
else:
    to_label.to_csv(OUT, index=False, encoding="utf-8-sig")
    print("저장:", OUT, len(to_label))

summary = pd.crosstab(to_label["split"], to_label["pool"], margins=True)
summary

기존 to_label.csv 유지 (2000건) | 같은 seed로 재샘플링한 결과와 동일: True


pool,boost,random,All
split,,,
test,0,300,300
train,1000,500,1500
val,0,200,200
All,1000,1000,2000


In [6]:
# 분할별 게임 분포·추천 비율 — Test가 원본 분포를 잘 대표하는지 확인
pd.DataFrame({
    "전체(정제)": df["voted_up"].mean(),
    "Train": to_label.query("split=='train'")["voted_up"].mean(),
    "Val":   to_label.query("split=='val'")["voted_up"].mean(),
    "Test":  to_label.query("split=='test'")["voted_up"].mean(),
}, index=["추천 비율"]).round(3)

,전체(정제),Train,Val,Test
추천 비율,0.833,0.79,0.85,0.837


## 다음 단계

`to_label.csv`의 2,000건에 `docs/labeling_guide.md` 기준으로 라벨을 붙인다 → `data/labeling/labels_llm.csv`